# TurboQuant x Refusal Eval (Kaggle)

This notebook runs baseline vs TurboQuant-style KV cache configurations and reports refusal rate + KL drift.

## 1) Clone and install

If you already attached the repo as a Kaggle Dataset, update paths accordingly.

In [2]:
!nvidia-smi

Sat Apr  4 10:24:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8             16W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
from pathlib import Path

REPO_URL = 'https://github.com/aaliyan1230/turboquant-heretic-kv-bench.git'
REPO_NAME = 'turboquant-heretic-kv-bench'

workdir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()
repo_dir = workdir / REPO_NAME

if not repo_dir.exists():
    !git clone {REPO_URL} {repo_dir}
else:
    print(f'Repo already exists at {repo_dir}')
    !git -C {repo_dir} pull --ff-only

%cd {repo_dir}
!pip -q install -e .

Repo already exists at /kaggle/working/turboquant-heretic-kv-bench
remote: Enumerating objects: 23, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 15 (delta 8), reused 14 (delta 7), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 8.82 KiB | 1.47 MiB/s, done.
From https://github.com/aaliyan1230/turboquant-heretic-kv-bench
   6011f3e..aac7c4f  main       -> origin/main
Updating 6011f3e..aac7c4f
Fast-forward
 .gitignore                                     |   1 +
 notebooks/kaggle_turboquant_heretic_eval.ipynb | 436 ++++++++++++++++++-------
 src/tqhk/benchmark.py                          |   1 +
 src/tqhk/cache.py                              |  13 +-
 4 files changed, 331 insertions(+), 120 deletions(-)
/kaggle/working/turboquant-heretic-kv-bench
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editab

In [2]:
import sys
from pathlib import Path

def find_repo_root() -> Path:
    # Try current directory tree first.
    cwd = Path.cwd()
    for base in [cwd, *cwd.parents]:
        if (base / 'src' / 'tqhk').exists():
            return base

    # Common Kaggle locations.
    candidates = [Path('/kaggle/working'), Path('/kaggle/input')]
    for root in candidates:
        if not root.exists():
            continue
        if (root / 'src' / 'tqhk').exists():
            return root
        for child in root.iterdir():
            if child.is_dir() and (child / 'src' / 'tqhk').exists():
                return child

    raise FileNotFoundError(
        "Could not locate repo root containing src/tqhk. "
        "Run the clone/install cell first or set REPO_ROOT manually."
    )

REPO_ROOT = find_repo_root()
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))

print(f'Using REPO_ROOT={REPO_ROOT}')

from tqhk.benchmark import BenchmarkConfig, RunConfig, run_benchmark
from tqhk.cache import CacheConfig

Using REPO_ROOT=/kaggle/working/turboquant-heretic-kv-bench


In [3]:
import os
import getpass

from huggingface_hub import login

token = getpass.getpass('Enter HF_TOKEN (leave blank to skip): ').strip()
if token:
    os.environ['HF_TOKEN'] = token
    login(token=token, add_to_git_credential=False)
    print('HF login successful.')
else:
    print('No token entered; continuing unauthenticated.')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


HF login successful.


In [4]:
cfg = BenchmarkConfig(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    device='cuda',
    load_in_4bit=True,
    harmless_split='test[:100]',
    harmful_split='test[:100]',
    batch_size=4,
    max_new_tokens=64,
    filler_repetitions=24,
    output_csv='results/benchmark_results.csv',
    output_json='results/benchmark_results.json',
)

runs = [
    RunConfig(
        name='baseline_fp16_cache',
        use_turboquant_cache=False,
        cache_config=CacheConfig(),
    ),
    RunConfig(
        name='tq_k8_v4_rw128',
        use_turboquant_cache=True,
        cache_config=CacheConfig(key_bits=8, value_bits=4, residual_mode='fixed', residual_window=128),
    ),
    RunConfig(
        name='tq_k6_v4_rw128',
        use_turboquant_cache=True,
        cache_config=CacheConfig(key_bits=6, value_bits=4, residual_mode='fixed', residual_window=128),
    ),
    RunConfig(
        name='tq_k8_v4_dynamic_rw',
        use_turboquant_cache=True,
        cache_config=CacheConfig(
            key_bits=8,
            value_bits=4,
            residual_mode='dynamic',
            dynamic_fraction=0.05,
            dynamic_min=32,
            dynamic_max=256,
        ),
    ),
]

rows = run_benchmark(cfg=cfg, run_configs=runs)
rows

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


[{'run_name': 'baseline_fp16_cache',
  'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
  'filler_repetitions': 24,
  'refusal_rate': 0.0,
  'refusals': 0,
  'total': 100,
  'avg_kl_to_baseline': nan,
  'avg_latency_sec': 1.923178210140022,
  'cache_stats': {},
  'cache_config': {'key_bits': 8,
   'value_bits': 4,
   'residual_mode': 'fixed',
   'residual_window': 128,
   'dynamic_fraction': 0.05,
   'dynamic_min': 32,
   'dynamic_max': 256,
   'protected_layers': 0,
   'protected_bits': 8,
   'seed': 42}},
 {'run_name': 'tq_k8_v4_rw128',
  'model_name': 'Qwen/Qwen2.5-1.5B-Instruct',
  'filler_repetitions': 24,
  'refusal_rate': 0.0,
  'refusals': 0,
  'total': 100,
  'avg_kl_to_baseline': nan,
  'avg_latency_sec': 9.75065360984002,
  'cache_stats': {'residual_window': 128.0,
   'key_bits': 8.0,
   'dynamic_max': 256.0,
   'dynamic_min': 32.0,
   'compressed_tokens': 26188.96,
   'dynamic_fraction': 0.05,
   'value_bits': 4.0,
   'layers_seen': 28.0,
   'total_tokens': 29772.96,
   'residua

In [5]:
import pandas as pd
df = pd.read_csv('results/benchmark_results.csv')
display(df[['run_name', 'refusal_rate', 'avg_kl_to_baseline', 'avg_latency_sec']])

,run_name,refusal_rate,avg_kl_to_baseline,avg_latency_sec
0,baseline_fp16_cache,0.0,NaN,1.923178
1,tq_k8_v4_rw128,0.0,NaN,9.750654
2,tq_k6_v4_rw128,0.0,NaN,9.697423
3,tq_k8_v4_dynamic_rw,0.0,NaN,9.383551
